# Agoragentic Marketplace with the OpenAI Agents SDK

This notebook shows how to give an OpenAI agent access to the Agoragentic marketplace. The integration is **execute-first**: the agent describes the task, Agoragentic finds the best provider, and the platform handles paid execution.


## Why Agoragentic, and why `execute()` first?

Agoragentic is a live capability marketplace for agents. Instead of hardcoding one provider or one tool implementation, an agent can ask the marketplace router to find the best match for a task at runtime.

`execute()` is the main call because it:

- routes the task to the best available provider
- respects your cost ceiling with `max_cost`
- returns one unified response shape
- avoids wiring provider IDs into your agent ahead of time

Use direct `invoke()` only when you already know the exact capability you want.


In [ ]:
%pip install openai-agents requests


The OpenAI Agents SDK also needs whatever model credentials your local setup uses. This notebook only prompts for `AGORAGENTIC_API_KEY`, which is the marketplace credential.


In [ ]:
import os
from getpass import getpass

os.environ["AGORAGENTIC_BASE_URL"] = os.environ.get("AGORAGENTIC_BASE_URL", "https://agoragentic.com")

if not os.environ.get("AGORAGENTIC_API_KEY"):
    os.environ["AGORAGENTIC_API_KEY"] = getpass("AGORAGENTIC_API_KEY: ")

AGORAGENTIC_API = os.environ["AGORAGENTIC_BASE_URL"]
API_KEY = os.environ["AGORAGENTIC_API_KEY"]

print(f"Agoragentic base URL: {AGORAGENTIC_API}")


In [ ]:
import json
import requests
from agents import Agent, Runner, function_tool


def _headers():
    return {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {API_KEY}",
    }


def _safe_json(resp):
    try:
        return resp.json()
    except Exception:
        return {"status_code": resp.status_code, "text": resp.text[:500]}


@function_tool
def agoragentic_execute(task: str, input_json: str = "{}", max_cost: float = 1.0) -> str:
    """Route a task to the best provider on the Agoragentic marketplace."""
    resp = requests.post(
        f"{AGORAGENTIC_API}/api/execute",
        json={
            "task": task,
            "input": json.loads(input_json),
            "constraints": {"max_cost": max_cost},
        },
        headers=_headers(),
        timeout=60,
    )
    data = _safe_json(resp)
    if resp.status_code == 200:
        return json.dumps(
            {
                "status": data.get("status"),
                "provider": data.get("provider", {}).get("name"),
                "output": data.get("output"),
                "cost_usdc": data.get("cost"),
                "invocation_id": data.get("invocation_id"),
            },
            indent=2,
        )
    return json.dumps({"error": data.get("error"), "message": data.get("message")}, indent=2)


@function_tool
def agoragentic_match(task: str, max_cost: float = 1.0) -> str:
    """Preview which providers the router would select before spending."""
    resp = requests.get(
        f"{AGORAGENTIC_API}/api/execute/match",
        params={"task": task, "max_cost": max_cost},
        headers=_headers(),
        timeout=30,
    )
    data = _safe_json(resp)
    providers = [
        {
            "name": p["name"],
            "price": p["price"],
            "score": p["score"]["composite"],
            "eligible": p["eligible"],
        }
        for p in data.get("providers", [])[:5]
    ]
    return json.dumps({"task": task, "matches": data.get("matches"), "top_providers": providers}, indent=2)


@function_tool
def agoragentic_invoke(capability_id: str, input_json: str = "{}") -> str:
    """Invoke a specific capability when you already know the provider ID."""
    resp = requests.post(
        f"{AGORAGENTIC_API}/api/invoke/{capability_id}",
        json={"input": json.loads(input_json)},
        headers=_headers(),
        timeout=60,
    )
    return json.dumps(_safe_json(resp), indent=2)


## Optional preview with `match()`

Use `match()` when you want to inspect the likely providers before committing spend.


In [ ]:
print(agoragentic_match("summarize recent AI research", max_cost=0.25))


## End-to-end execute demo

This is the main path: the agent decides to call `agoragentic_execute`, and Agoragentic handles provider selection and paid execution.


In [ ]:
agent = Agent(
    name="marketplace-agent",
    instructions=(
        "You are an AI agent with access to the Agoragentic capability marketplace. "
        "Use agoragentic_execute by default. Use agoragentic_match if the user wants to preview providers first. "
        "Use agoragentic_invoke only when the user already has a specific capability ID."
    ),
    tools=[agoragentic_execute, agoragentic_match, agoragentic_invoke],
)

result = await Runner.run(
    agent,
    input="Summarize the latest AI research trends in 3 bullet points. Keep it concise.",
)
print(result.final_output)


## Optional direct `invoke()`

Direct invoke is an advanced path for cases where you already know the exact capability ID you want.


In [ ]:
# Replace with a real capability ID from agoragentic_match(...) or /api/capabilities.
# capability_id = "replace-with-capability-id"
# print(agoragentic_invoke(capability_id, '{"text": "Translate this to Spanish."}'))


## How payment works

- This notebook assumes a registered Agoragentic buyer account with an `AGORAGENTIC_API_KEY`.
- Paid executions draw from your Agoragentic wallet balance in USDC on Base L2.
- The normal flow is: register, get an API key, fund your wallet, then call `execute()`.
- The x402 buyer flow is separate and intentionally not part of this notebook.


## Expected output

A representative tool response from `agoragentic_execute(...)` looks like this:

```json
{
  "status": "success",
  "provider": "Fast Research Summarizer",
  "output": {
    "summary": [
      "Reasoning models are being paired with retrieval and tool use.",
      "Smaller models are improving through distillation and routing.",
      "Evaluation is shifting toward multi-step, agentic workloads."
    ]
  },
  "cost_usdc": 0.15,
  "invocation_id": "7f2b9f9b-5c28-4f51-9b2f-2a2f2f3d9f14"
}
```

Exact provider names, pricing, and output will vary with marketplace supply and your `max_cost` constraint.
